# Exercise 4 — Joint posterior $(\Omega_m,\, H_0 r_d)$ from DESI BAO via MCMC

**Course:** Computational Astrophysics and Cosmology — University of Roma Tre (Master's degree)

**Course topics covered:**
- 2D Metropolis-Hastings MCMC with flat priors.
- Cosmological distances in flat $\Lambda$CDM: $D_C$, $D_M$, $D_A$, $D_H$, $D_V$.
- Natural parameterization of BAO: fitting $(\Omega_m, H_0 r_d)$ without fixing the sound horizon.
- Multi-measurement BAO likelihood with isotropic ($D_V/r_d$), transverse ($D_M/r_d$), and line-of-sight ($D_H/r_d$) observables.
- *(Optional)* Extension to alternative dark-energy cosmologies.

**Reference papers (DESI 2024 Year 1):**
- DESI Collaboration, 2024, *AJ* **168**, 58 — *DESI 2024 I: A Panchromatic View of the Universe with a Wide-Area Spectroscopic Survey* — [arXiv:2306.06308](https://arxiv.org/abs/2306.06308).
- DESI Collaboration, 2024 — *DESI 2024 III: Baryon Acoustic Oscillations from Galaxies and Quasars* — [arXiv:2404.03002](https://arxiv.org/abs/2404.03002).  Table 1 contains the final BAO measurements for each tracer.
- DESI Collaboration, 2024 — *DESI 2024 IV: Baryon Acoustic Oscillations from the Lyman Alpha Forest* — [arXiv:2404.03001](https://arxiv.org/abs/2404.03001).  Table 1 contains the Ly$\alpha$ BAO measurement.
- DESI Collaboration, 2024 — *DESI 2024 VI: Cosmological Constraints from the Measurements of Baryon Acoustic Oscillations* — [arXiv:2404.03000](https://arxiv.org/abs/2404.03000).  This is the primary reference for cosmological interpretation.

**Dataset:** BAO measurements from the DESI Year 1 (2024) spectroscopic survey.  
The student is expected to retrieve the numerical values from the tables of the papers listed above before running this notebook.  The cells below indicate precisely which table and which columns to extract.

**Parameterization.** Flat $\Lambda$CDM.  Free MCMC parameters: $(\Omega_m,\; H_0 r_d)$.  No external prior on $r_d$ is needed: all BAO observables $D/r_d$ depend only on $(\Omega_m, H_0 r_d)$.  If an external measurement of $r_d$ is available (e.g. from CMB or BBN), $H_0$ can be derived *a posteriori*.

**Scientific scenario — what BAO alone constrains.**

BAO use the sound horizon $r_d$ at the drag epoch as a *standard ruler*. Measuring $D_M(z)/r_d$, $D_H(z)/r_d$, or $D_V(z)/r_d$ at multiple redshifts constrains the expansion history $H(z)$.  In flat $\Lambda$CDM the two free parameters are $\Omega_m$ (shape of $H(z)/H_0$) and the combination $H_0 r_d$ (overall amplitude, since $D \propto c/(H_0 r_d)$).  Neither $H_0$ nor $r_d$ alone is accessible from BAO without an external anchor.  Fixing $r_d$ from Planck 2018 ($r_d = 147.78$ Mpc, Aghanim+2020) converts the BAO constraint on $H_0 r_d$ into a constraint on $H_0 \approx 68$ km/s/Mpc — on the CMB side of the **Hubble tension**.

In [12]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import minimize

try:
    import corner
    HAS_CORNER = True
except ImportError:
    HAS_CORNER = False
    print("corner not available — using histogram2d fallback.")

np.random.seed(42)
c = 299792.458  # km/s

corner not available — using histogram2d fallback.


## Task 1 — Retrieve and enter DESI BAO measurements

Read **Table 1** of [arXiv:2404.03002](https://arxiv.org/abs/2404.03002) (galaxy/QSO tracers) and **Table 1** of [arXiv:2404.03001](https://arxiv.org/abs/2404.03001) (Ly$\alpha$ tracer).  For each measurement note:

- The effective redshift $z_\mathrm{eff}$.
- The observable type:
  - `"DV"` — isotropic, $D_V(z)/r_d$
  - `"DM"` — transverse, $D_M(z)/r_d$
  - `"DH"` — line-of-sight, $D_H(z)/r_d$ where $D_H(z) \equiv c/H(z)$
- The measured value and $1\sigma$ uncertainty.
- The tracer name (BGS, LRG1, …, Ly$\alpha$).

DESI 2024 provides **seven** tracers.  For each tracer that yields an anisotropic split ($D_M$ + $D_H$) there is also a correlation coefficient $\rho$ between the two measurements — enter it for the optional covariance extension (Task 7).

Enter the values in the code cell below by replacing the `???` placeholders.

In [ ]:
# ── Task 1: DESI 2024 BAO measurements ───────────────────────────────────────
# Read Table 1 of arXiv:2404.03002 (galaxy + QSO tracers)
# and Table 1 of arXiv:2404.03001 (Lya tracer).
#
# Each row: (z_eff, value, sigma, type, tracer_name)
#   type ∈ {"DV", "DM", "DH"}
#
# Replace every None with the numerical value from the paper.

bao_data = np.array([
    # BGS — isotropic (Table 1, arXiv:2404.03002)
    (0.295, None, None, "DV", "DESI-BGS"),
    # LRG Sample 1 — anisotropic (Table 1, arXiv:2404.03002)
    (0.510, None, None, "DM", "DESI-LRG1"),
    (0.510, None, None, "DH", "DESI-LRG1"),
    # LRG Sample 2 — anisotropic (Table 1, arXiv:2404.03002)
    (0.706, None, None, "DM", "DESI-LRG2"),
    (0.706, None, None, "DH", "DESI-LRG2"),
    # LRG3 + ELG1 combined — anisotropic (Table 1, arXiv:2404.03002)
    (0.930, None, None, "DM", "DESI-LRG3/ELG1"),
    (0.930, None, None, "DH", "DESI-LRG3/ELG1"),
    # ELG Sample 2 — anisotropic (Table 1, arXiv:2404.03002)
    (1.317, None, None, "DM", "DESI-ELG2"),
    (1.317, None, None, "DH", "DESI-ELG2"),
    # QSO — isotropic (Table 1, arXiv:2404.03002)
    (1.491, None, None, "DV", "DESI-QSO"),
    # Lya — anisotropic (Table 1, arXiv:2404.03001)
    (2.330, None, None, "DM", "DESI-Lya"),
    (2.330, None, None, "DH", "DESI-Lya"),
], dtype=object)

# DM–DH cross-correlation coefficients ρ (same tables; values are negative, ~-0.4)
rho_DM_DH = {
    "DESI-LRG1":       None,   # TODO: read from Table 1, arXiv:2404.03002
    "DESI-LRG2":       None,
    "DESI-LRG3/ELG1":  None,
    "DESI-ELG2":       None,
    "DESI-Lya":        None,   # TODO: read from Table 1, arXiv:2404.03001
}

print(f"DESI BAO table: {len(bao_data)} measurements from {len(set(bao_data[:,4]))} tracers")
print(f"{'z':>6}  {'obs':>6}  {'value':>8}  {'sigma':>7}  tracer")
print("-" * 50)
for row in bao_data:
    z, val, sig, typ, name = row
    print(f"{float(z):6.3f}  {typ:>6}  {str(val):>8}  {str(sig):>7}  {name}")

## Task 2 — Theoretical model: distances in $(\Omega_m,\, H_0 r_d)$ space

### Why this parameterization?

In flat $\Lambda$CDM:

$$E(z) \equiv \frac{H(z)}{H_0} = \sqrt{\Omega_m(1+z)^3 + (1-\Omega_m)}$$

The comoving distance is $D_C(z) = (c/H_0)\,\tilde I(z,\Omega_m)$ with $\tilde I = \int_0^z dz'/E(z')$ dimensionless. Hence:

$$\frac{D_M(z)}{r_d} = \frac{c}{H_0 r_d}\,\tilde I(z,\Omega_m)$$

$$\frac{D_H(z)}{r_d} \equiv \frac{c}{H(z)\,r_d} = \frac{c}{H_0 r_d}\cdot\frac{1}{E(z)}$$

$$\frac{D_V(z)}{r_d} = \frac{c}{H_0 r_d}\left[z\,\tilde I^2(z,\Omega_m)\,\frac{1}{E(z)}\right]^{1/3}$$

All three observables depend **only** on $(\Omega_m,\, H_0 r_d)$.  Fitting BAO therefore constrains this pair directly without needing to know $r_d$ or $H_0$ separately.

We define `H0_rd` $\equiv H_0 r_d$ in km/s as the second MCMC parameter (range $\sim 6000$–$15000$ km/s, corresponding to $H_0\in[50,100]$ km/s/Mpc and $r_d\in[130,160]$ Mpc).

In [ ]:
# ── Task 2: distance functions in (Omega_m, H0_rd) space ─────────────────────

def E(z, Om):
    """E(z) = H(z)/H0 for flat ΛCDM."""
    # TODO: return sqrt(Om*(1+z)^3 + (1-Om))
    pass

def I_z(z, Om):
    """Dimensionless comoving integral ∫_0^z dz'/E(z')."""
    # TODO: use scipy.integrate.quad
    pass

def DM_over_rd(z, Om, H0_rd):
    """Transverse comoving distance / r_d."""
    # TODO: (c / H0_rd) * I_z(z, Om)
    pass

def DH_over_rd(z, Om, H0_rd):
    """Hubble distance c/H(z) / r_d."""
    # TODO: (c / H0_rd) / E(z, Om)
    pass

def DV_over_rd(z, Om, H0_rd):
    """Volume-averaged distance / r_d (Eisenstein+2005)."""
    # TODO: (c / H0_rd) * (z * I_z(z,Om)^2 / E(z,Om))^(1/3)
    pass

# Sanity check — run after implementing the functions above.
# Expected values at (Om=0.30, H0=68 km/s/Mpc, rd=147.78 Mpc):
Om_fid   = 0.30
rd_fid   = 147.78   # Mpc — Planck 2018
H0_fid   = 68.0     # km/s/Mpc
H0rd_fid = H0_fid * rd_fid

print(f"Fiducial: Om={Om_fid}, H0={H0_fid} km/s/Mpc, rd={rd_fid} Mpc")
print(f"  => H0_rd = {H0rd_fid:.1f} km/s")
print(f"  DM/rd(0.51)  = {DM_over_rd(0.510,  Om_fid, H0rd_fid)}  (target ~13.6)")
print(f"  DH/rd(0.51)  = {DH_over_rd(0.510,  Om_fid, H0rd_fid)}  (target ~21.0)")
print(f"  DV/rd(0.295) = {DV_over_rd(0.295, Om_fid, H0rd_fid)}  (target ~7.9)")

## Task 3 — BAO likelihood

Assuming each measurement is Gaussian and the measurements are uncorrelated (diagonal covariance):

$$\chi^2_\mathrm{BAO}(\Omega_m, H_0 r_d)
  = \sum_i\left(\frac{\mathrm{obs}_i - \mathrm{model}_i(\Omega_m, H_0 r_d)}{\sigma_i}\right)^2$$

where $\mathrm{model}_i$ is $D_V/r_d$, $D_M/r_d$, or $D_H/r_d$ according to the `type` column.

**Note on correlations.** For each DESI tracer that provides both $D_M/r_d$ and $D_H/r_d$, these two quantities are correlated (cross-spectrum; the correlation coefficient $\rho$ is given in the paper table).  In the basic version here we ignore these off-diagonal terms; Task 7 (optional) shows how to include them via a $2\times 2$ block covariance.

In [ ]:
# ── Task 3: BAO data vector and full covariance matrix ────────────────────────

_obs_func = {"DV": DV_over_rd, "DM": DM_over_rd, "DH": DH_over_rd}

# Build the full n×n covariance matrix.
# Start from the diagonal (σ_i²), then fill in the off-diagonal blocks
# for each (DM, DH) pair belonging to the same tracer using:
#   C[i, j] = ρ * σ_i * σ_j

n_bao  = len(bao_data)
sigmas = np.array([float(row[2]) for row in bao_data])

# TODO: initialize C_BAO as a diagonal matrix with entries σ_i²
C_BAO = None  # replace with: np.diag(sigmas**2)

# TODO: loop over all (i, j) pairs with i ≠ j.
#   Fill C_BAO[i, j] = rho_DM_DH[name] * σ_i * σ_j
#   when rows i and j share the same tracer name AND have types {"DM", "DH"}.

# TODO: invert C_BAO
C_BAO_inv = None  # replace with: np.linalg.inv(C_BAO)

# Optional sanity check: print the correlation matrix
# D_diag = np.sqrt(np.diag(C_BAO))
# corr = C_BAO / np.outer(D_diag, D_diag)
# print(np.round(corr, 3))

# Observation vector (do not modify)
_obs_vec = np.array([float(row[1]) for row in bao_data])

def model_vector(Om, H0_rd, data=bao_data):
    """Return the 12-element vector of BAO model predictions."""
    # TODO: for each row compute _obs_func[typ](z, Om, H0_rd) and collect into array
    pass

def chi2_BAO(Om, H0_rd):
    """χ² with the full covariance matrix: r^T C^{-1} r."""
    # TODO: r = _obs_vec - model_vector(Om, H0_rd)
    # TODO: return float(r @ C_BAO_inv @ r)
    pass

def log_like_BAO(Om, H0_rd):
    return -0.5 * chi2_BAO(Om, H0_rd)

# Sanity check (needs Tasks 1 and 2 complete first)
print(f"chi2_BAO(0.30, H0rd_fid) = {chi2_BAO(0.30, H0rd_fid)}")
print(f"chi2_BAO(0.30, 12000)    = {chi2_BAO(0.30, 12000.0)}  (should be much larger)")

## Task 4 — 2D Metropolis-Hastings MCMC

Standard algorithm:
1. Current point $\theta_n = (\Omega_m, H_0 r_d)$, log-likelihood $\ell_n$.
2. Propose $\theta' = \theta_n + \mathcal{N}(0, \Sigma_\mathrm{prop})$.
3. Reject if $\theta'$ falls outside the priors.
4. Accept/reject via Metropolis criterion $\ln u < \ell' - \ell_n$ with $u \sim U(0,1)$.

**Flat priors:** $\Omega_m \in [0.1, 0.6]$, $H_0 r_d \in [6000, 15000]$ km/s.

**Proposal:** $\sigma_{\Omega_m} \approx 0.02$, $\sigma_{H_0 r_d} \approx 150$ km/s.  Target acceptance rate $0.2$–$0.4$ in 2D.

In [ ]:
# ── Task 4: Metropolis-Hastings ───────────────────────────────────────────────

def metropolis_2d(log_like, theta0, prop_cov, n_steps, prior_bounds):
    """
    N-dimensional Metropolis-Hastings sampler with flat priors.

    Parameters
    ----------
    log_like     : callable — log_like(theta[0], theta[1], ...) -> float
    theta0       : array (ndim,) — starting point
    prop_cov     : array (ndim, ndim) — Gaussian proposal covariance
    n_steps      : int — total number of steps (including burn-in)
    prior_bounds : list of (lo, hi) tuples, one per dimension

    Returns
    -------
    chain        : array (n_steps, ndim)
    acc_rate     : float — fraction of accepted proposals
    """
    ndim = len(theta0)
    # TODO: allocate chain = np.zeros((n_steps, ndim))
    # TODO: chain[0] = theta0; cur_ll = log_like(*theta0); n_acc = 0
    # TODO: for i in range(1, n_steps):
    #   prop = chain[i-1] + np.random.multivariate_normal(np.zeros(ndim), prop_cov)
    #   if any prop[k] outside prior_bounds[k]: chain[i] = chain[i-1]; continue
    #   new_ll = log_like(*prop)
    #   if np.log(np.random.rand()) < new_ll - cur_ll:
    #       chain[i] = prop; cur_ll = new_ll; n_acc += 1
    #   else:
    #       chain[i] = chain[i-1]
    # TODO: return chain, n_acc / n_steps
    pass

## Task 5 — Run the MCMC and inspect the chain

Run a single chain of 60 000 steps (burn-in: 10 000 discarded).  Check:
- Acceptance rate in $[0.2, 0.4]$.
- Trace plots of $\Omega_m$ and $H_0 r_d$ look well-mixed after burn-in.
- The chain concentrates around physically reasonable values ($\Omega_m \sim 0.3$, $H_0 r_d \sim 10\,000$ km/s).

In [ ]:
# ── Task 5: run MCMC ──────────────────────────────────────────────────────────

prior_bounds = [(0.10, 0.60), (6000.0, 15000.0)]  # (Omega_m, H0_rd [km/s])
prop_cov     = np.diag([0.02**2, 150.0**2])
theta0       = np.array([0.30, H0rd_fid])
n_steps      = 60000
burn         = 10000

# TODO: call metropolis_2d and store results
# chain_raw, acc = metropolis_2d(log_like_BAO, theta0, prop_cov, n_steps, prior_bounds)
# print(f"  acceptance rate = {acc:.3f}")   # target: 0.20 – 0.40

# TODO: discard burn-in
# chain = chain_raw[burn:]
# print(f"  post-burn-in samples: {len(chain)}")

# TODO: make trace plots (2 rows, 1 column; share the x-axis)
# For each parameter k plot chain_raw[:, k] vs step index (lw=0.3).
# Mark the burn-in boundary with a red dashed vertical line.
# Labels: [r"$\Omega_m$", r"$H_0 r_d$ [km/s]"]

## Task 6 — Posterior contours in $(\Omega_m,\, H_0 r_d)$

Plot the 68% and 95% confidence regions from the MCMC chain.  Also show:
- The marginalized 1D posteriors of both parameters.
- Lines of constant $H_0$ (for $r_d = 147.78$ Mpc) overlaid on the 2D contour — this shows how a CMB prior on $r_d$ converts the BAO constraint on $H_0 r_d$ into one on $H_0$.

What to expect:
- $\Omega_m$ is well constrained ($\sim 0.30 \pm 0.02$) — it controls the *shape* of $H(z)/H_0$.
- $H_0 r_d$ is tightly constrained ($\sim 10\,000 \pm 150$ km/s) — it controls the *amplitude* of all distances.
- With $r_d = 147.78$ Mpc fixed, the derived $H_0 = (H_0 r_d)/r_d \approx 68$ km/s/Mpc.

In [ ]:
# ── Task 6: posterior contours ────────────────────────────────────────────────

def levels_from_hist(H, levels=(0.68, 0.95)):
    """Return the histogram bin heights that enclose fractions `levels` of the density."""
    Hs  = np.sort(H.flatten())[::-1]
    cum = np.cumsum(Hs) / Hs.sum()
    out = []
    for lv in levels:
        idx = np.searchsorted(cum, lv)
        out.append(Hs[min(idx, len(Hs)-1)])
    return sorted(out)

# TODO: produce a 2D posterior plot of (Omega_m, H0_rd) at 68% and 95% levels.
#
# If HAS_CORNER is True: call corner.corner(chain, ...) with
#   labels=[r"$\Omega_m$", r"$H_0\,r_d\;[\mathrm{km/s}]$"],
#   quantiles=[0.16, 0.5, 0.84], show_titles=True, levels=(0.68, 0.95).
#
# Otherwise: use np.histogram2d + scipy.ndimage.gaussian_filter (sigma=1.2)
#   then plt.contourf / plt.contour; use levels_from_hist() to find the
#   correct iso-density levels.
#
# In both cases overplot horizontal lines at H0 = 66, 68, 70, 72 km/s/Mpc
#   (i.e. H0_rd = H0 * rd_fid) on the 2D panel.
#
# Add a second panel with the marginalized H0 posterior:
#   H0_derived = chain[:, 1] / rd_fid
#   annotate the median and ±1σ (16th/84th percentiles).
#
# Save the figure as "posterior_Om_H0rd.png".

## Task 7 — Data vs Theory: DESI 2024 measurements and model predictions

Plot all 12 BAO measurements with their $1\sigma$ error bars and overlay the theoretical curves $D_M/r_d(z)$, $D_H/r_d(z)$, $D_V/r_d(z)$ evaluated at two parameter sets:

- **Planck fiducial**: $(\Omega_m, H_0 r_d) = (0.30,\; 68 \times 147.78)$ km/s — the CMB-derived baseline.
- **MCMC median**: the posterior median of $(\Omega_m, H_0 r_d)$ derived from the DESI data themselves.

The two panels separate the *anisotropic* measurements ($D_M/r_d$ and $D_H/r_d$) from the *isotropic* ones ($D_V/r_d$). The agreement between data and theory in both panels validates the likelihood and the MCMC chain.

In [ ]:
# ── Task 7: Data vs Theory ────────────────────────────────────────────────────

# Compute posterior medians from the MCMC chain
# Om_mcmc   = np.median(chain[:, 0])
# H0rd_mcmc = np.median(chain[:, 1])

# Planck fiducial reference point
# Om_pl   = 0.30
# H0rd_pl = H0rd_fid   # = 68.0 * 147.78 km/s

# TODO: build z_fine = np.linspace(0.10, 2.60, 500)

# TODO: compute smooth theory curves (one per observable type) at both parameter sets:
#   DM_pl, DH_pl, DV_pl  at (Om_pl,   H0rd_pl)
#   DM_mc, DH_mc, DV_mc  at (Om_mcmc, H0rd_mcmc)
# using the functions DM_over_rd, DH_over_rd, DV_over_rd from Task 2.

# TODO: split bao_data into three lists (DM rows, DH rows, DV rows),
#   each collecting (z, value, sigma, tracer_name).

# TODO: create a figure with two side-by-side panels.
#
# Left panel — anisotropic BAO:
#   plot DM theory curves (solid = Planck, dashed = MCMC) in one color (e.g. royalblue)
#   plot DH theory curves in a second color (e.g. firebrick)
#   overlay data errorbars with markers (squares for DM, triangles for DH)
#   annotate each data point with the tracer name
#
# Right panel — isotropic BAO:
#   plot DV theory curves (solid + dashed) in a third color (e.g. seagreen)
#   overlay DV data errorbars with diamond markers
#   annotate each data point with the tracer name
#
# Add a suptitle showing both parameter sets.
# Save as "data_vs_theory.png".

## Task 8 — Summary: save the percentiles

Save to `posterior_summary.txt` the median and $\pm 1\sigma$ (16th/84th percentiles) for both sampled parameters and the derived $H_0$ (using $r_d^\mathrm{Planck} = 147.78$ Mpc).

In [ ]:
# ── Task 8: save percentiles ──────────────────────────────────────────────────

# TODO: compute the derived H0 posterior from the MCMC chain
#   H0_derived = chain[:, 1] / rd_fid

# TODO: stack into chain_ext = np.column_stack([chain, H0_derived])
#   so that column 0 = Omega_m, column 1 = H0_rd, column 2 = H0_derived

# TODO: for each of the three quantities compute np.percentile(..., [16, 50, 84])
#   and format as: "param_name = median  (-lo / +hi)"

# TODO: write the result to "posterior_summary.txt" and print it to stdout

## Validation: comparison with ML (`scipy.optimize.minimize`)

The maximum-likelihood point must fall inside the 1$\sigma$ MCMC contour.  If it does not, re-check the data table, the likelihood, or the proposal covariance.

In [ ]:
# Validation: ML via Nelder-Mead
#
# Use scipy.optimize.minimize (method="Nelder-Mead") to find the
# maximum-likelihood point independently of the MCMC chain.
# The ML estimate must agree with the MCMC median to better than 1%.

# TODO: define neg_log_like(theta) returning -log_like_BAO(*theta)
#   (return a large value, e.g. 1e10, if theta is outside the prior)
# TODO: call minimize(neg_log_like, x0=[0.30, H0rd_fid], method="Nelder-Mead")
# TODO: print ML estimate and MCMC median side by side and compute relative difference

---
## (Optional) Task 9 — Alternative cosmologies

The MCMC framework above is entirely general: any modification to $E(z)$ can be plugged in.  **Implement one (or both) extensions below and compare the posteriors with the flat $\Lambda$CDM result.**

---

### 9a — Dynamical dark energy: $w_0 w_a$CDM (Chevallier–Polarski–Linder)

The dark-energy equation of state varies with scale factor $a=1/(1+z)$:
$$w(a) = w_0 + w_a(1-a) = w_0 + w_a\frac{z}{1+z}$$

The Friedmann equation becomes:
$$E^2(z) = \Omega_m(1+z)^3 + (1-\Omega_m)(1+z)^{3(1+w_0+w_a)}
           \exp\!\left(-\frac{3w_a z}{1+z}\right)$$

Free parameters for MCMC: $(\Omega_m,\, H_0 r_d,\, w_0,\, w_a)$.

- Priors: $w_0 \in [-2, 0]$, $w_a \in [-3, 2]$.
- Note: the MCMC is now 4D — increase `n_steps` and tune `prop_cov` accordingly.
- **DESI 2024 found a $2.5\sigma$ preference for $w_0 > -1$** (or $w_a < 0$).  Can you see it in your chain?

---

### 9b — Non-flat $\Lambda$CDM

Add a curvature component $\Omega_k = 1 - \Omega_m - \Omega_\Lambda$:
$$E^2(z) = \Omega_m(1+z)^3 + \Omega_k(1+z)^2 + (1 - \Omega_m - \Omega_k)$$

For non-flat geometry $D_M \neq D_C$:
$$D_M = \frac{c}{H_0}\cdot\frac{1}{\sqrt{|\Omega_k|}}\begin{cases}
  \sinh\!\left(\sqrt{|\Omega_k|}\,\tilde I\right) & \Omega_k>0\\
  \sin\!\left(\sqrt{|\Omega_k|}\,\tilde I\right)  & \Omega_k<0
\end{cases}$$

Free parameters: $(\Omega_m,\, H_0 r_d,\, \Omega_k)$.  Prior: $\Omega_k \in [-0.5, 0.5]$.

In [ ]:
# ── Optional Task 10a: w0waCDM ────────────────────────────────────────────────

def E_w0wa(z, Om, w0, wa):
    """E(z) for flat w0waCDM (Chevallier–Polarski–Linder)."""
    # TODO: de_exp = (1+z)^{3(1+w0+wa)} * exp(-3*wa*z/(1+z))
    # TODO: return sqrt(Om*(1+z)^3 + (1-Om)*de_exp)
    pass

def DM_over_rd_w0wa(z, Om, H0_rd, w0, wa):
    # TODO: integral of 1/E_w0wa, same pattern as DM_over_rd
    pass

def DH_over_rd_w0wa(z, Om, H0_rd, w0, wa):
    # TODO: (c/H0_rd) / E_w0wa(z, Om, w0, wa)
    pass

def DV_over_rd_w0wa(z, Om, H0_rd, w0, wa):
    # TODO: same structure as DV_over_rd but using E_w0wa
    pass

_obs_func_w0wa = {
    "DV": DV_over_rd_w0wa,
    "DM": DM_over_rd_w0wa,
    "DH": DH_over_rd_w0wa,
}

def chi2_BAO_w0wa(Om, H0_rd, w0, wa):
    # TODO: compute model vector using _obs_func_w0wa for each row of bao_data
    # TODO: return float(r @ C_BAO_inv @ r)  — reuse the same covariance matrix
    pass

def log_like_w0wa(Om, H0_rd, w0, wa):
    return -0.5 * chi2_BAO_w0wa(Om, H0_rd, w0, wa)

# ── 4D MCMC ──
prior_bounds_w0wa = [
    (0.10,   0.60),    # Omega_m
    (6000., 15000.),   # H0_rd [km/s]
    (-2.0,   0.0),     # w0
    (-3.0,   2.0),     # wa
]
prop_cov_w0wa = np.diag([0.02**2, 150.0**2, 0.05**2, 0.2**2])
theta0_w0wa   = np.array([0.30, H0rd_fid, -1.0, 0.0])

# TODO: call metropolis_2d(log_like_w0wa, theta0_w0wa, prop_cov_w0wa,
#                          n_steps, prior_bounds_w0wa)
# TODO: discard burn-in; print acceptance rate and median w0, wa
# TODO: if HAS_CORNER, produce a 4-panel corner plot and save as "posterior_w0wa.png"